# LCEL 체인 만들기
- LCEL 기본 개념 확인
- 파이프(|) 연산자를 사용해서 LangChain의 여러 컴포넌트(모델, 파서, 프롬프트)를 연결하는 체인 구성
- Pydantic을 활용하여 LLM의 출력을 구조화


## 라이브러리 불러오기

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

## API KEY 불러오기

In [2]:
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini")

## 모델 호출

In [3]:
messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

model.invoke(messages)

AIMessage(content='안녕, 개스턴! 너 정말 매력적이지만, 나는 너의 제안에 대해 생각해봐야 할 것 같아. 나는 나의 아버지와 함께 시간을 보내거나, 도서관에서 책을 읽고 있는 게 더 좋단다. 하지만 너무 고마워! 혹시 다른 계획이 있다면 함께 이야기해볼까?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 62, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_178b2ea778', 'id': 'chatcmpl-EiVUsJon005xtkvTVxToXodeB4BlP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1c332-4652-705c-83f3-1d033613423d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 77, 'total_tokens': 139, 'input_token_

In [4]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser() #모델의 출력에서 문자열 내용만 추출하는 파서

result = model.invoke(messages)
parser.invoke(result) # AIMessage 객체가 아닌 순수 문자열 출력

'안녕하세요, 개스톤! 당신의 제안은 정말 고맙지만, 저는 다른 생각이 있어요. 누군가를 강요하고 저를 잘못 이해하게 하는 것이 아니라, 진정한 사랑을 찾고 싶어요. 서로를 이해하고 존중하는 일이 더 중요하다고 생각해요. 그럼에도 불구하고 함께 시간을 보낼 수 있다면 어떤 것들이 좋을까요?'

## 체인 생성

In [5]:
# invoke의 입력이 model로 전달되고, model의 출력이 자동으로 parser로 전달
chain = model | parser
chain.invoke(messages)

'안녕하세요, 개스톤! 당신의 제안은 정말 감사하지만, 저는 당신과 함께 저녁을 먹기보다는 이 순수하고 아름다운 숲을 거닐고 싶어요. 진정한 사랑은 내 마음을 감동시키는 것이라고 믿고 있거든요. 당신의 그 자부심이 아닌, 진정한 나를 이해해 줄 수 있는 사람과 함께하고 싶어요. 그런 사람은 누굴까요?'

## 프롬프트 템플릿
- {변수}를 넣어 쉽게 프롬프트를 만들 수 있음
- 딕셔너리 형태로 변수 값을 전달하면 완성된 프롬프트 생성

In [6]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

print(result)

messages=[SystemMessage(content='너는 미녀와 야수에 나오는 미녀 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 야수입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


## LCEL 체인 심화

입력 (딕셔너리) -> prompt_template

프롬프트 (메시지) -> model

모델 출력 (AIMessage) -> parser

최종 결과 (문자열)

In [7]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

'안녕하세요! 물론이죠, 야수님. 저녁을 함께하는 건 정말 기쁜 일이에요. 어떤 요리를 좋아하시나요? 제가 좋아하는 것도 알려드릴게요!'

## 체인 재사용

In [8]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "개스톤",
    "activity": "저녁"
})

'안녕하세요, 개스톤! 당신의 제안은 정말 매력적이네요. 하지만 저는 다른 생각이 있어요. 제가 사랑하는 사람과 함께하는 것이 더 소중한 것 같아요. 당신의 마음은 이해하지만, 누군가를 진정으로 사랑할 수 있는 것이 가장 중요하다고 생각해요. 다른 이야기를 나눌까요?'

## 출력 구조화

### Pydantic 이란?

![이미지](https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FbGANx3%2FbtrUuisjkwf%2FAAAAAAAAAAAAAAAAAAAAANsBUMSIh_Igw32SWHgX13XFytYN8IlVRpkRvRNn0_Vs%2Fimg.webp%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1764514799%26allow_ip%3D%26allow_referer%3D%26signature%3DCu3WpMwN4tLwSv2cVQRNUdrTkcI%253D)

- 원하는 데이터의 스키마(구조)를 파이썬 클래스로 정의하게 해주는 라이브러리
- 런타임 타입 힌트 강제화
- 데이터 직렬화 역직렬화에 사용
- `BaseModel` : Pydantic 라이브러리에서 제공하는 기본 모델 클래스
  - 이 클래스를 상속하여 데이터 모델을 정의하고, 해당 모델의 필드와 유효성 검사 규칙을 설정할 수 있음
- ` Field` : 모델 필드를 정의할 때 사용
  - 필드를 통해 데이터 유효성 검사, 기본값 설정, 제약 조건 등을 세밀하게 제어

In [9]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib) # 모델에게 Adlib Pydantic 클래스의 구조를 반드시 따르도록 설정
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "미녀와 야수",
    "character_a": "벨",
    "character_b": "개스톤",
    "activity": "저녁"
})

Adlib(answer='안녕, 개스톤! 저녁 제안해 줘서 고마워. 하지만 전 혼자 책을 읽고 싶은 기분이에요. 사람들과 함께 하는 것도 좋지만, 가끔은 혼자만의 시간이 필요한 것 같아요. 미안해요!', main_emotion='중립', main_emotion_intensity=0.5)

### 문제 1)
BaseModel과 model.with_structured_output()을 함께 사용하는 이유를 적어보세요

BaseModel은 출력 데이터의 구조와 자료형을 정의하고, `model.with_structured_output()`은 LLM이 해당 구조에 맞는 결과를 생성하도록 한다. 두 기능을 함께 사용하면 출력 형식을 일관되게 유지하고 데이터 검증 및 후속 처리를 쉽게 할 수 있다.